# Train Height CNN

Phase 2.2 of [3D_plan1.md](../docs/3D_plan1.md): train the U-Net building-height
predictor on paired (satellite-RGB, height-raster) tiles collected from Phase-1
ground-truth providers.

**Pipeline overview**
1. **Collect tiles** — grid each city's bbox, fetch height rasters from providers,
   discard tiles with >50% NaN.
2. **Train** — U-Net (EfficientNet-B4 encoder), L1 + Sobel gradient loss, AdamW,
   cosine-annealed LR.  Best checkpoint saved automatically.
3. **Evaluate** — compute MAE on the validation split.
4. **Use** — load checkpoint into `predict_heights(model="unet", checkpoint=...)`,
   or use the session helper.

**Prerequisites**
```bash
pip install torch torchvision timm transformers
# optional for the smp encoder:
pip install segmentation-models-pytorch
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from app.session.terrain_session import TerrainSession
from app.server.core.height.train import (
    collect_tiles,
    train,
    TrainConfig,
    TileDataset,
    _DEFAULT_CITIES,
)

PROJECT_ROOT = Path('..').resolve()
TILE_DIR = PROJECT_ROOT / 'cache' / 'height_tiles'
CHECKPOINT = PROJECT_ROOT / 'models' / 'height_unet.pt'
print(f"Tile dir : {TILE_DIR}")
print(f"Checkpoint: {CHECKPOINT}")

## 1. Configure training run

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Configuration — edit these parameters.
# ──────────────────────────────────────────────────────────────────────────────

CITIES = ["Barcelona", "Granada"]    # Cities to collect tiles from
PROVIDERS = ["ndsm", "wsf3d"]        # Height providers for labels
TILES_PER_CITY = 200                 # Tiles collected per city

EPOCHS = 50                          # Training epochs
BATCH_SIZE = 8                       # Mini-batch size
LEARNING_RATE = 1e-4                 # Initial LR (cosine-annealed to LR*0.01)
GRAD_WEIGHT = 0.5                    # Weight for Sobel gradient loss
DEVICE = "cpu"                       # "cpu" or "cuda"

print(f"Cities    : {CITIES}")
print(f"Providers : {PROVIDERS}")
print(f"Epochs    : {EPOCHS}")
print(f"Device    : {DEVICE}")

print("\nAvailable cities:")
for k, v in _DEFAULT_CITIES.items():
    print(f"  {k:12s} N={v.north:.3f} S={v.south:.3f} E={v.east:.3f} W={v.west:.3f}")

## 2. Collect tiles

Grid the city bboxes, fetch height rasters from the chosen providers, and save
paired (RGB placeholder, height) tiles to `cache/height_tiles/`.

> **Note**: Satellite RGB tiles are saved as zeros until satellite fetch is
> wired.  The height labels are real.  Training with zero RGB will not produce
> a useful model — this step demonstrates the pipeline structure.  To produce
> a trained model, populate the RGB arrays with actual satellite imagery.

In [ ]:
print("Collecting tiles…")
tile_paths = collect_tiles(
    CITIES,
    tile_dir=TILE_DIR,
    providers=PROVIDERS,
    tiles_per_city=TILES_PER_CITY,
)
print(f"Total tiles collected: {len(tile_paths)}")

### Inspect a sample tile

In [ ]:
if tile_paths:
    sample = np.load(tile_paths[0])
    rgb = sample["rgb"]      # (3, H, W) float32
    height = sample["height"] # (1, H, W) float32

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(rgb.transpose(1, 2, 0))  # (H, W, 3)
    axes[0].set_title("RGB tile (placeholder)")
    axes[0].axis("off")
    im = axes[1].imshow(height[0], cmap="hot")
    axes[1].set_title("Height tile (metres)")
    axes[1].axis("off")
    plt.colorbar(im, ax=axes[1], label="m")
    plt.tight_layout()
    plt.show()
    print(f"Height range: [{height.min():.1f}, {height.max():.1f}] m")
else:
    print("No tiles to display")

## 3. Train

In [ ]:
if not tile_paths:
    print("No tiles — skipping training")
else:
    cfg = TrainConfig(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE,
        grad_weight=GRAD_WEIGHT,
        device=DEVICE,
    )

    print(f"Training: {cfg.epochs} epochs, batch={cfg.batch_size}, lr={cfg.lr}, device={DEVICE}")
    metrics = train(tile_paths, CHECKPOINT, cfg)

    print()
    print(f"  Best val loss : {metrics['best_val_loss']:.4f}")
    print(f"  Epochs trained: {metrics['epochs_trained']}")
    print(f"  Train tiles   : {metrics['n_train']}")
    print(f"  Val tiles     : {metrics['n_val']}")
    print(f"  Checkpoint    : {metrics['checkpoint']}")

## 4. Evaluate on validation split

In [ ]:
if tile_paths and CHECKPOINT.exists():
    import torch
    from app.server.core.height.predict import _build_unet
    from torch.utils.data import DataLoader, random_split

    full_ds = TileDataset(tile_paths, augment=False)
    n_val = max(1, int(len(full_ds) * cfg.val_split))
    n_train = len(full_ds) - n_val
    _, val_ds = random_split(
        full_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(cfg.seed)
    )
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, num_workers=0)

    state = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    model = _build_unet(pretrained_encoder=False)
    model.load_state_dict(state["model_state_dict"])
    model.eval()

    all_preds = []
    all_targets = []
    with torch.no_grad():
        for rgb, height in val_loader:
            pred = model(rgb)
            all_preds.append(pred.numpy())
            all_targets.append(height.numpy())

    preds = np.concatenate(all_preds).ravel()
    targets = np.concatenate(all_targets).ravel()
    mae = np.mean(np.abs(preds - targets))
    rmse = np.sqrt(np.mean((preds - targets) ** 2))
    print(f"Validation MAE  : {mae:.2f} m")
    print(f"Validation RMSE : {rmse:.2f} m")

    fig, ax = plt.subplots(figsize=(6, 6))
    lim = max(targets.max(), preds.max()) * 1.05
    ax.scatter(targets, preds, alpha=0.1, s=2, c="steelblue")
    ax.plot([0, lim], [0, lim], "r--", lw=1)
    ax.set_xlabel("Ground truth height (m)")
    ax.set_ylabel("Predicted height (m)")
    ax.set_title(f"U-Net validation  (MAE={mae:.2f} m, RMSE={rmse:.2f} m)")
    plt.tight_layout()
    plt.savefig('../output/height_unet_eval.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No tiles or checkpoint — skipping evaluation")

## 5. Use the trained model via TerrainSession

In [ ]:
if CHECKPOINT.exists():
    BBOX = dict(north=41.395, south=41.375, east=2.175, west=2.145)
    s = TerrainSession(bbox=BBOX)
    s.fetch_satellite(zoom=17)

    s.predict_heights(model="unet", checkpoint=str(CHECKPOINT), device=DEVICE)
    r = s.predicted_heights
    print(f"Predicted height range: [{r.raster.min():.1f}, {r.raster.max():.1f}] m")
else:
    print("No checkpoint yet — run training first")

## 6. Re-train using the TerrainSession helper (alternative)

In [ ]:
# This is equivalent to the cells above but uses the session method.
# Uncomment to run:

# s2 = TerrainSession(bbox=dict(north=41.42, south=41.35, east=2.19, west=2.12))
# result = s2.train_height_model(
#     cities=["Barcelona", "Granada"],
#     epochs=50,
#     batch_size=8,
#     device="cpu",
# )
# print(result)